In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import mapping
import xarray as xr

from climakitae.core.data_interface import DataParameters


def open_shapefile_and_clip(shapefile_path, data) -> tuple[xr.DataArray, gpd.GeoDataFrame]:
    """
    Open a shapefile and clip the data to the shapefile's geometry.

    Parameters:
        shapefile_path (str): Path to the shapefile.
        data (xarray.DataArray): Data to be clipped.
        crs (cartopy.crs.Projection): Coordinate reference system for the shapefile.

    Returns:
        xarray.DataArray: Clipped data.
    """
    # Load the shapefile
    gdf = gpd.read_file(shapefile_path)

    # Clip the data using the shapefile's geometry
    clipped_data = data.rio.clip(gdf.geometry.apply(mapping), gdf.crs)

    return clipped_data, gdf

In [ ]:

shapefile_path = "./examples/PajaroRiverWatershed.zip"
selections = DataParameters()
selections.approach = "Warming Level"
selections.warming_level_window = 10
selections.warming_level = [2.0]
selections.downscaling_method = "Dynamical"
selections.variable = "Minimum air temperature at 2m"
selections.timescale = "hourly"
selections.units = "degF"

print("Loading data...")
da = selections.retrieve()

# median nighttime temperature
print("Calculating median nighttime temperature...")
clipped_data, gdf = open_shapefile_and_clip(shapefile_path, da)

In [ ]:
# show dimensions for analysis
print("Data dimensions:")
print(clipped_data.dims)

# show some values for time_delta
print("Data values:")
print(clipped_data)

print(clipped_data.centered_year)

In [ ]:
# First, check the type of time_delta
print("Time delta type:", type(clipped_data.time_delta.values[0]))
print("Sample values:", clipped_data.time_delta.values[:5])

# Assuming time_delta is hours offset from a reference time
reference_time = np.datetime64('2039-01-01')  # Adjust this if needed
time_as_datetime = pd.to_datetime(reference_time) + pd.to_timedelta(clipped_data.time_delta.values, unit='h')

# convert to PST
time_as_datetime = time_as_datetime.tz_localize('UTC').tz_convert('America/Los_Angeles')
print("Converted time values:")
print(time_as_datetime[:5])

In [ ]:

# Create a new coordinate with datetime values
clipped_data = clipped_data.assign_coords(datetime=('time_delta', time_as_datetime))

# Create a DataArray for is_night with the same dimensions as the time dimension
hours = pd.DatetimeIndex(clipped_data.datetime.values).hour
is_night = ((hours >= 21) | (hours <= 5))
is_night_da = xr.DataArray(is_night, dims='time_delta', coords={'time_delta': clipped_data.time_delta})

# Filter data to keep only nighttime values
nighttime = clipped_data.where(is_night_da, drop=True)

# Calculate daily median of nighttime temperatures
daily_median_nighttime = nighttime.resample(datetime="1D").median()

# First compute the median across time dimension
median_over_time = nighttime.median(dim="time_delta")

In [ ]:
# Import ProgressBar from dask.diagnostics
from dask.diagnostics.progress import ProgressBar

# Select first simulation and compute with progress bar
print("Computing data with progress bar...")
with ProgressBar():
    plot_data = median_over_time.isel(simulation=0).compute()

    


In [ ]:
# plot with Cartopy
print("Plotting data...")
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection=ccrs.PlateCarree())

plot_data = plot_data.squeeze()  # Remove any singleton dimensions

# Plot using pcolormesh with proper coordinate handling
print("meshing data...")
mesh = ax.pcolormesh(
    plot_data.lon,
    plot_data.lat,
    plot_data,
    cmap="RdYlBu_r",
    transform=ccrs.PlateCarree(),
    shading="auto",  # Automatically handles grid registration
)

# Add colorbar
print("adding colorbar...")
cbar = plt.colorbar(
    mesh,
    ax=ax,
    orientation="horizontal",
    pad=0.05,
    aspect=40,
    label="Temperature (°F)",
)

# Enhanced map features
print("adding map features...")
ax.add_feature(cfeature.STATES.with_scale("10m"), edgecolor="black", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)

# Set extent to Pajaro region (adjusted coordinates)
ax.set_extent([-122.2, -120.5, 36.2, 37.4], crs=ccrs.PlateCarree())

# Add watershed boundary with transparency
print("adding watershed boundary...")
gdf_projected = gdf.to_crs(ccrs.PlateCarree().proj4_init)
gdf_projected.plot(ax=ax, facecolor="none", edgecolor="darkgreen", linewidth=1.5, alpha=0.7)

# Add grid lines
print("adding grid lines...")
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.5,
    color="gray",
    alpha=0.5,
    linestyle="--",
)
gl.top_labels = False
gl.right_labels = False

# Title with improved formatting
plt.title(
    "Median Minimum Temperature Under 2.0°C GWL\nPajaro Watershed (°F)",
    fontsize=14,
    pad=20,
    weight="bold",
)

# Adjust layout and save
plt.tight_layout()
plt.savefig(
    "pajaro_night_temp_2C_GWL_F.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()